In [1]:
# NORTHSTAR — Solace Browser: User Onboarding Flow QA
# DNA: onboarding_qa = entry_points(start+home) x wizard(5_steps) x tutorial(5_steps) x js_quality(no_var+escapeHtml) x a11y(aria+focus) x i18n(47_locales)
# Template: solace-cli/notebooks/qa-templates/template-flow-qa.ipynb
# Towers: T1(Masters) T2(Customers) T6(Hackers) T8(Elders) T9(World) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads HTML+JS source directly
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-onboarding-flow-qa"
NOTEBOOK_PATH = "notebooks/qa/30-onboarding-flow-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
JS_DIR = WEB / "js"

# Onboarding source files
ONBOARDING_JS = JS_DIR / "onboarding.js"
SETUP_WIZARD_JS = JS_DIR / "setup-wizard.js"
TUTORIAL_V1_JS = JS_DIR / "yinyang-tutorial.js"
TUTORIAL_V2_JS = JS_DIR / "yinyang-tutorial-v2.js"
START_HTML = WEB / "start.html"
HOME_HTML = WEB / "home.html"
GUIDE_HTML = WEB / "guide.html"

ALL_ONBOARDING_FILES = [ONBOARDING_JS, SETUP_WIZARD_JS, TUTORIAL_V1_JS, TUTORIAL_V2_JS]
ALL_HTML_FILES = [START_HTML, HOME_HTML, GUIDE_HTML]

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

# Verify all files exist
missing = [str(f) for f in ALL_ONBOARDING_FILES + ALL_HTML_FILES if not f.exists()]
print(f"BOOTSTRAP: {len(ALL_ONBOARDING_FILES)} JS files, {len(ALL_HTML_FILES)} HTML files")
print(f"Missing files: {missing if missing else 'none'}")
print(f"Files: {[f.name for f in ALL_ONBOARDING_FILES + ALL_HTML_FILES]}")

BOOTSTRAP: 4 JS files, 3 HTML files
Missing files: none
Files: ['onboarding.js', 'setup-wizard.js', 'yinyang-tutorial.js', 'yinyang-tutorial-v2.js', 'start.html', 'home.html', 'guide.html']


In [2]:
# P1: All onboarding JS files exist and have no var declarations
print("=== P1: Onboarding JS Files Exist + No var ===")

for js_file in ALL_ONBOARDING_FILES:
    exists = js_file.exists()
    record(f"P1-exists-{js_file.name}", f"{js_file.name} exists", exists,
           detail=f"File not found: {js_file}" if not exists else "",
           tower_ref="T1,T23")

    if exists:
        content = js_file.read_text(encoding="utf-8")
        # Check for var declarations (should use const/let instead)
        # Exclude comments and strings: simple heuristic — lines starting with var
        var_lines = []
        for i, line in enumerate(content.split("\n"), 1):
            stripped = line.lstrip()
            if stripped.startswith("//") or stripped.startswith("*"):
                continue
            if re.search(r'\bvar\s+\w+', stripped):
                var_lines.append((i, stripped[:80]))

        record(f"P1-no-var-{js_file.name}", f"{js_file.name} has no var declarations ({len(var_lines)} found)",
               len(var_lines) == 0,
               detail=f"var found at lines: {[v[0] for v in var_lines[:5]]}" if var_lines else "",
               tower_ref="T1,T16")

passed = sum(1 for r in results if r["id"].startswith("P1") and r["status"] == "PASS")
total_p1 = sum(1 for r in results if r["id"].startswith("P1"))
print(f"\nP1 complete: {passed}/{total_p1} passed")

=== P1: Onboarding JS Files Exist + No var ===
  PASS: onboarding.js exists
  PASS: onboarding.js has no var declarations (0 found)
  PASS: setup-wizard.js exists
  PASS: setup-wizard.js has no var declarations (0 found)
  PASS: yinyang-tutorial.js exists
  PASS: yinyang-tutorial.js has no var declarations (0 found)
  PASS: yinyang-tutorial-v2.js exists
  PASS: yinyang-tutorial-v2.js has no var declarations (0 found)

P1 complete: 8/8 passed


In [3]:
# P2: Setup wizard has 5+ steps with progress indicator
print("=== P2: Setup Wizard Steps + Progress ===")

if SETUP_WIZARD_JS.exists():
    wizard_src = SETUP_WIZARD_JS.read_text(encoding="utf-8")

    # Find STEPS array or step definitions
    steps_match = re.search(r'const\s+STEPS\s*=\s*\[', wizard_src)
    record("P2-steps-array", "setup-wizard.js defines STEPS array",
           steps_match is not None,
           detail="No STEPS array found" if not steps_match else "",
           tower_ref="T1,T2")

    # Count step objects in STEPS array (look for { title: or { id: patterns)
    step_objects = re.findall(r'\{\s*(?:title|id|label|name)\s*:', wizard_src)
    record("P2-step-count", f"setup-wizard.js has 5+ wizard steps ({len(step_objects)} found)",
           len(step_objects) >= 5,
           detail=f"Only {len(step_objects)} step objects found, need >=5" if len(step_objects) < 5 else "",
           tower_ref="T1,T2")

    # Progress indicator (progress bar, dots, step counter)
    has_progress = bool(re.search(r'progress|wiz-dot|step\s*\d+\s*of|currentStep', wizard_src, re.IGNORECASE))
    record("P2-progress-indicator", "setup-wizard.js has progress indicator",
           has_progress,
           detail="No progress indicator (dots, bar, or step counter)" if not has_progress else "",
           tower_ref="T1,T2,T8")

    # Skip option (steps can be skipped)
    has_skip = bool(re.search(r'skip|optional|can be skipped', wizard_src, re.IGNORECASE))
    record("P2-skip-option", "setup-wizard.js allows skipping steps",
           has_skip,
           detail="No skip/optional mechanism found" if not has_skip else "",
           tower_ref="T1,T2")
else:
    record("P2-wizard-missing", "setup-wizard.js exists", False,
           detail="File not found", tower_ref="T1")

passed = sum(1 for r in results if r["id"].startswith("P2") and r["status"] == "PASS")
total_p2 = sum(1 for r in results if r["id"].startswith("P2"))
print(f"\nP2 complete: {passed}/{total_p2} passed")

=== P2: Setup Wizard Steps + Progress ===
  PASS: setup-wizard.js defines STEPS array
  PASS: setup-wizard.js has 5+ wizard steps (11 found)
  PASS: setup-wizard.js has progress indicator
  PASS: setup-wizard.js allows skipping steps

P2 complete: 4/4 passed


In [4]:
# P3: Tutorial has step-by-step flow with next/back navigation
print("=== P3: Tutorial Navigation (Next/Back) ===")

tutorial_files = [TUTORIAL_V1_JS, TUTORIAL_V2_JS]
any_tutorial_ok = False

for tut_file in tutorial_files:
    if not tut_file.exists():
        record(f"P3-exists-{tut_file.name}", f"{tut_file.name} exists", False,
               detail=f"File not found: {tut_file}", tower_ref="T1")
        continue

    tut_src = tut_file.read_text(encoding="utf-8")

    # Step-by-step flow (step1, step2, ... or TOTAL_STEPS or step array)
    has_steps = bool(re.search(r'step\d+|TOTAL_STEPS|_currentStep|currentStep|stepHTML', tut_src))
    record(f"P3-steps-{tut_file.name}", f"{tut_file.name} has step-based flow",
           has_steps,
           detail="No step-based flow found" if not has_steps else "",
           tower_ref="T1,T2")

    # Next button
    has_next = bool(re.search(r'[Nn]ext|btn_next|nextStep|_next', tut_src))
    record(f"P3-next-{tut_file.name}", f"{tut_file.name} has Next navigation",
           has_next,
           detail="No Next button/function found" if not has_next else "",
           tower_ref="T1,T2,T8")

    # Back/Previous button
    has_back = bool(re.search(r'[Bb]ack|[Pp]rev|btn_prev|prevStep|_prev', tut_src))
    record(f"P3-back-{tut_file.name}", f"{tut_file.name} has Back navigation",
           has_back,
           detail="No Back/Previous button/function found" if not has_back else "",
           tower_ref="T1,T2,T8")

    if has_steps and has_next:
        any_tutorial_ok = True

record("P3-any-tutorial", "At least one tutorial has complete step navigation",
       any_tutorial_ok,
       detail="Neither tutorial has working step + next flow" if not any_tutorial_ok else "",
       tower_ref="T1,T2")

passed = sum(1 for r in results if r["id"].startswith("P3") and r["status"] == "PASS")
total_p3 = sum(1 for r in results if r["id"].startswith("P3"))
print(f"\nP3 complete: {passed}/{total_p3} passed")

=== P3: Tutorial Navigation (Next/Back) ===
  PASS: yinyang-tutorial.js has step-based flow
  PASS: yinyang-tutorial.js has Next navigation
  PASS: yinyang-tutorial.js has Back navigation
  PASS: yinyang-tutorial-v2.js has step-based flow
  PASS: yinyang-tutorial-v2.js has Next navigation
  PASS: yinyang-tutorial-v2.js has Back navigation
  PASS: At least one tutorial has complete step navigation

P3 complete: 7/7 passed


In [5]:
# P4: Entry points (start.html, home.html, guide.html) connect to onboarding
print("=== P4: Entry Points Connect to Onboarding ===")

ONBOARDING_SCRIPTS = ["onboarding.js", "setup-wizard.js", "yinyang-tutorial.js", "yinyang-tutorial-v2.js"]

for html_file in ALL_HTML_FILES:
    if not html_file.exists():
        record(f"P4-exists-{html_file.name}", f"{html_file.name} exists", False,
               detail=f"File not found: {html_file}", tower_ref="T1")
        continue

    html_src = html_file.read_text(encoding="utf-8")

    # Check which onboarding scripts are referenced
    referenced = []
    for script in ONBOARDING_SCRIPTS:
        if re.search(re.escape(script), html_src):
            referenced.append(script)

    # guide.html is the onboarding content itself (step-by-step sections with guide-num)
    # so it doesn't need to reference the JS scripts — it IS the guide
    is_guide_content = bool(re.search(r'class="guide-num"', html_src))

    record(f"P4-scripts-{html_file.name}",
           f"{html_file.name} {'has onboarding content' if is_guide_content else 'references onboarding scripts: ' + str(referenced)}",
           len(referenced) >= 1 or is_guide_content,
           detail=f"No onboarding scripts or guide content in {html_file.name}" if not (referenced or is_guide_content) else "",
           tower_ref="T1,T2")

# Specifically check start.html loads setup-wizard
if START_HTML.exists():
    start_src = START_HTML.read_text(encoding="utf-8")
    has_wizard = bool(re.search(r'setup-wizard\.js', start_src))
    record("P4-start-wizard", "start.html loads setup-wizard.js",
           has_wizard,
           detail="start.html does not reference setup-wizard.js" if not has_wizard else "",
           tower_ref="T1,T2")

passed = sum(1 for r in results if r["id"].startswith("P4") and r["status"] == "PASS")
total_p4 = sum(1 for r in results if r["id"].startswith("P4"))
print(f"\nP4 complete: {passed}/{total_p4} passed")

=== P4: Entry Points Connect to Onboarding ===
  PASS: start.html references onboarding scripts: ['setup-wizard.js', 'yinyang-tutorial-v2.js']
  PASS: home.html references onboarding scripts: ['onboarding.js']
  PASS: guide.html has onboarding content
  PASS: start.html loads setup-wizard.js

P4 complete: 4/4 passed


In [6]:
# P5: Evidence summary — aggregate results, compute score, seal hash
print("=== P5: Evidence Summary ===")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = round(100 * passed / total, 1) if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"  NOTEBOOK: {NOTEBOOK_PATH}")
print(f"  PROJECT:  {PROJECT}")
print(f"  JS FILES: {[f.name for f in ALL_ONBOARDING_FILES]}")
print(f"  HTML:     {[f.name for f in ALL_HTML_FILES]}")
print(f"  TOTAL:    {total} probes")
print(f"  PASSED:   {passed}")
print(f"  FAILED:   {failed}")
print(f"  SCORE:    {score}%  (min required: {MIN_SCORE}%)")
print(f"{'='*60}")

if failed > 0:
    print("\nFAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['id']} — {r['name']}")
            if r["detail"]:
                print(f"        {r['detail']}")

# Seal
evidence = {
    "notebook": NOTEBOOK_PATH,
    "northstar": NORTHSTAR,
    "project": PROJECT,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "total": total,
    "passed": passed,
    "failed": failed,
    "score_pct": score,
    "min_score": MIN_SCORE,
    "verdict": "PASS" if score >= MIN_SCORE else "FAIL",
    "results": results,
}

seal = hashlib.sha256(json.dumps(evidence, sort_keys=True).encode()).hexdigest()[:16]
evidence["seal"] = seal

print(f"\n  VERDICT: {evidence['verdict']}")
print(f"  SEAL:    {seal}")
print(f"  AUTH:    65537")

=== P5: Evidence Summary ===

  NOTEBOOK: notebooks/qa/30-onboarding-flow-qa.ipynb
  PROJECT:  solace-browser
  JS FILES: ['onboarding.js', 'setup-wizard.js', 'yinyang-tutorial.js', 'yinyang-tutorial-v2.js']
  HTML:     ['start.html', 'home.html', 'guide.html']
  TOTAL:    23 probes
  PASSED:   23
  FAILED:   0
  SCORE:    100.0%  (min required: 70%)

  VERDICT: PASS
  SEAL:    d77fc2768bc3979e
  AUTH:    65537
